# Milestone 4

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import numpy as np
import os
import glob
import random
from pathlib import Path
from torch.utils.data import DataLoader, Dataset



INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)

def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
              "jazz", "metal", "pop", "reggae", "rock"]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")

print("Helper functions loaded successfully!")

Helper functions loaded successfully!


In [3]:
# Q1: Number of .wav files in synthetic dataset
# samples_per_genre=50, 10 genres, so 50 * 10 = 500 files

samples_per_genre = 50
num_genres = 10
total_files = samples_per_genre * num_genres

print(f"Q1: Total .wav files in /kaggle/working/synthetic_mashups/train/: {total_files}")

Q1: Total .wav files in /kaggle/working/synthetic_mashups/train/: 500


In [4]:
# Q2: Tensor shape of loaded waveform
# target_sr=22050, duration=30
# Shape: (channels, samples) = (2, 22050 * 30) = (2, 661500)
# But torchaudio.load() might return (1, 661500) for mono

target_sr = 22050
duration = 30
expected_samples = target_sr * duration

print(f"Q2: Expected tensor shape: (1, {expected_samples})")
print(f"Q2: In tuple format: (1, {expected_samples})")

Q2: Expected tensor shape: (1, 661500)
Q2: In tuple format: (1, 661500)


In [5]:
# Q3: Shape of pre-computed Mel-spectrogram tensor
# n_fft=2048, hop_length=512, n_mels=128
# Input: 30 seconds at 22050 Hz = 661500 samples
# Time frames: (661500 - 2048) / 512 + 1 = 1291
# Output shape: (n_mels, time_frames) = (128, 1291)

n_fft = 2048
hop_length = 512
n_mels = 128
input_samples = 22050 * 30

time_frames = (input_samples - n_fft) // hop_length + 1
expected_shape = (n_mels, time_frames)

print(f"Q3: Mel-spectrogram tensor shape: {expected_shape}")

Q3: Mel-spectrogram tensor shape: (128, 1288)


In [6]:
# CRNN Model Implementation
class CRNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CRNN, self).__init__()
        
        # CNN Feature Extractor
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        
        # RNN
        self.lstm = nn.LSTM(input_size=2048, hidden_size=64, 
                          batch_first=True, bidirectional=True)
        
        # Final classifier
        self.fc = nn.Linear(128, num_classes)  # 64*2 for bidirectional
    
    def forward(self, x):
        # CNN blocks
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        
        # Store shape for Q4
        batch_size, channels, freq_bins, time_steps = x.shape
        
        # Reshape for RNN: (batch, time, features)
        x = x.permute(0, 3, 1, 2)  # (batch, time, channels, freq)
        x = x.reshape(batch_size, time_steps, -1)  # (batch, time, features)
        
        # RNN
        lstm_out, _ = self.lstm(x)
        
        # Global max pooling
        pooled, _ = torch.max(lstm_out, dim=1)
        
        # Final classification
        output = self.fc(pooled)
        
        return output

# Create model instance
model = CRNN(num_classes=10)
print("CRNN model created successfully!")

CRNN model created successfully!


In [7]:
# Q4: Shape after second MaxPool2d before permute
# Input: (batch_size, 1, 128, 1291)
# After Conv2d(1->32): (batch_size, 32, 128, 1291)
# After MaxPool2d(2): (batch_size, 32, 64, 645)
# After Conv2d(32->64): (batch_size, 64, 64, 645)
# After MaxPool2d(2): (batch_size, 64, 32, 322)

batch_size = 32
input_shape = (batch_size, 1, 128, 1291)

# Simulate the forward pass to get the shape
with torch.no_grad():
    dummy_input = torch.randn(*input_shape)
    x = model.conv_block1(dummy_input)
    x = model.conv_block2(x)
    shape_after_pool2 = x.shape

print(f"Q4: Shape after second MaxPool2d: {shape_after_pool2}")
print(f"Q4: For batch_size=32: (32, 64, 32, 322)")

Q4: Shape after second MaxPool2d: torch.Size([32, 64, 32, 322])
Q4: For batch_size=32: (32, 64, 32, 322)


In [8]:
# Q5: LSTM trainable parameters
# Bidirectional LSTM with input_size=2048, hidden_size=64
# Formula: 4 * (input_size + hidden_size) * hidden_size * num_directions
# + 4 * hidden_size * num_directions (bias terms)

lstm_params = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)

print(f"Q5: LSTM trainable parameters: {lstm_params}")

# Manual calculation verification
input_size = 2048
hidden_size = 64
num_directions = 2  # bidirectional

# Weight parameters: 4 * (input_size + hidden_size) * hidden_size * directions
weights = 4 * (input_size + hidden_size) * hidden_size * num_directions
# Bias parameters: 4 * hidden_size * directions  
biases = 4 * hidden_size * num_directions
total_manual = weights + biases

print(f"Q5: Manual calculation: {total_manual}")

Q5: LSTM trainable parameters: 1082368
Q5: Manual calculation: 1081856


In [9]:
# Training Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Training setup complete!")
print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Training setup complete!
Device: cpu
Model parameters: 1,102,666
